# Alba 2001 Ministry archive — structural completeness audit

This notebook runs or resumes the **source-structure audit only**. It does not estimate sorting, congestion, peer effects, or selection outcomes.

Its job is to determine whether the archived 2001 Alba reports can support trustworthy groups of participating applicants from the same originating gymnasium. It checks the source-provided school directory, exact within-school record correspondence, cross-county placements, and unresolved archive addresses.

**Cursor kernel:** select `/opt/anaconda3/envs/sports_net/bin/python` if Cursor asks you to choose an interpreter. The notebook prints the active Python path before any live retrieval.

**Privacy and provenance:** student names are used temporarily in memory for exact correspondence. They are never written to the audit outputs. Raw archived pages are not cached.

## 1. Review the run controls

The live-run switch is deliberately `False`. Review these settings first, then change only `RUN_LIVE_AUDIT` to `True` when ready.

The starting interval of 2 seconds is approximately 30 requests per minute. The retriever can cautiously shorten that interval after ten consecutive successes, down to 1.2 seconds, and lengthens it after any failure, up to 60 seconds. These are operational pacing choices, not scientific parameters.

In [ ]:
# SAFETY SWITCH: keep False while reviewing. Change to True to start/resume.
RUN_LIVE_AUDIT = False

# Adaptive request spacing, in seconds.
# Start moderately; speed up only after sustained success; slow after failures.
INITIAL_INTERVAL_SECONDS = 2.0
MINIMUM_INTERVAL_SECONDS = 1.2
MAXIMUM_INTERVAL_SECONDS = 60.0

# The complete notebook run will stop after this many hours even if unresolved
# addresses remain. All completed school checkpoints survive that stop.
MAX_RUNTIME_HOURS = 6.0

# After the first directory pass, revisit unresolved addresses this many times.
# A three-minute cooldown separates passes. A server Retry-After overrides this.
RETRY_PASSES = 3
RETRY_PASS_COOLDOWN_SECONDS = 180.0


## 2. Locate the project and load the audit driver

This cell works when the notebook is opened from its own folder or when Cursor starts the kernel from the repository root. It imports the existing program rather than duplicating the audit logic inside the notebook.

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

def find_education_root():
    """Find education_analysis without assuming Cursor's kernel working directory."""
    here = Path.cwd().resolve()
    candidates = []
    for anchor in [here, *here.parents]:
        # Case 1: the kernel starts at education_analysis or below it.
        candidates.append(anchor / "education_analysis")
        candidates.append(anchor)
        # Case 2: the kernel starts at the repository root or a parent.
        candidates.append(
            anchor
            / "3-Master_Plan"
            / "VECTOR_work"
            / "new_VECTOR_work"
            / "education_analysis"
        )
    for candidate in candidates:
        if (candidate / "code" / "EDUCATION_20260929_alba_origin_report_checks.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate education_analysis from the current working directory.")

EDUCATION_ROOT = find_education_root()
CODE_DIR = EDUCATION_ROOT / "code"
OUTPUT_DIR = EDUCATION_ROOT / "outputs" / "romania_alba_2001_structural_audit_20260929"

# Add only this audit's code directory to the import path.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

driver_path = CODE_DIR / "EDUCATION_20260929_alba_origin_report_checks.py"
spec = importlib.util.spec_from_file_location("alba_origin_audit", driver_path)
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)

print("Python:", sys.executable)
print("Education workspace:", EDUCATION_ROOT)
print("Audit driver:", driver_path)
print("Output directory:", OUTPUT_DIR)


## 3. Inspect the current checkpoint before requesting anything

The earlier interrupted pass already recovered some schools. The next cell reads only local checkpoint files and shows how many school codes are complete or unresolved. It makes no network requests.

In [ ]:
def show_checkpoint():
    """Print a compact, name-free view of the saved audit state."""
    summary_path = OUTPUT_DIR / "origin_report_check_summary.json"
    unresolved_path = OUTPUT_DIR / "unresolved_addresses.json"
    pace_path = OUTPUT_DIR / "retrieval_adaptive_state.json"
    live_progress_path = OUTPUT_DIR / "live_progress.json"
    checks_path = OUTPUT_DIR / "origin_report_checks.jsonl"

    # Derive resume progress directly from the append-only checkpoints, even
    # before the first consolidated summary has been written.
    latest = {}
    if checks_path.exists():
        for line in checks_path.read_text().splitlines():
            if line.strip():
                record = json.loads(line)
                latest[record["source_code"]] = record
    completed = sum(audit.is_complete(record) for record in latest.values())
    print(f"Saved school checkpoints: {len(latest)}; currently complete: {completed}")

    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        print("Current summary:")
        print(json.dumps(summary, indent=2, ensure_ascii=False))
    else:
        print("No current consolidated summary yet. Existing JSONL checkpoints will still be resumed.")

    if unresolved_path.exists():
        unresolved = json.loads(unresolved_path.read_text())
        print(f"\nUnresolved school codes: {len(unresolved)}")
        print("First five unresolved source codes:", [x["source_code"] for x in unresolved[:5]])

    if live_progress_path.exists():
        live_progress = json.loads(live_progress_path.read_text())
        print("\nMost recently saved live progress:")
        print(json.dumps(live_progress, indent=2, ensure_ascii=False))

    if pace_path.exists():
        pace = json.loads(pace_path.read_text())
        print("\nPersisted adaptive pacing state:")
        print(json.dumps(pace, indent=2, ensure_ascii=False))

show_checkpoint()


## 4. Run or resume

When `RUN_LIVE_AUDIT` is `True`, this cell runs automatically through the source directory, writes and flushes a checkpoint after every school, then revisits unresolved addresses in later passes.

Progress appears below the cell. The live bar shows the current pass, schools attempted in that pass, cumulative completed schools, unresolved schools, elapsed time, estimated time remaining, and the current request interval. Waits and backoffs are announced explicitly. The same status is refreshed in `live_progress.json` after every school, so progress survives outside the notebook display.

You may interrupt the kernel at any time. The cell closes the progress bar, writes the current summary and unresolved-address list, and then returns control to you. Rerunning the cell resumes from completed school codes.

In [ ]:
if not RUN_LIVE_AUDIT:
    print("DRY REVIEW: no archive requests were made.")
    print("Set RUN_LIVE_AUDIT = True in the first code cell when ready.")
else:
    try:
        audit.main(
            archive_use_acknowledged=True,
            initial_interval_seconds=INITIAL_INTERVAL_SECONDS,
            minimum_interval_seconds=MINIMUM_INTERVAL_SECONDS,
            maximum_interval_seconds=MAXIMUM_INTERVAL_SECONDS,
            max_runtime_hours=MAX_RUNTIME_HOURS,
            retry_passes=RETRY_PASSES,
            retry_pass_cooldown_seconds=RETRY_PASS_COOLDOWN_SECONDS,
        )
    except (audit.m.RetrievalStopped, KeyboardInterrupt) as error:
        # Preserve an honest stopping state whether the archive asks us to
        # stop, the maximum runtime is reached, or you interrupt the cell.
        reason = str(error) or "Interrupted by user"
        audit.m.emit({"kind": "stopped", "reason": reason})
        directory = json.loads((OUTPUT_DIR / "origin_school_directory.json").read_text())
        audit.write_summary(directory, stopped_reason=reason)
        print(f"Audit stopped safely: {reason}")
    finally:
        print("\nLatest saved state after this run:")
        show_checkpoint()


## 5. Read the stopping state

Run this cell after completion or interruption. `unresolved_addresses.json` is the authoritative list of addresses still requiring another attempt. An unresolved address is not treated as evidence that the historical page is absent.

In [ ]:
show_checkpoint()

# The detailed event stream provides request status, adaptive pace changes,
# waits, and stops without saving student-level rows.
events_path = OUTPUT_DIR / "events.jsonl"
if events_path.exists():
    event_lines = events_path.read_text().splitlines()
    print(f"\nRecorded audit events: {len(event_lines)}")
    print("Most recent five events:")
    for line in event_lines[-5:]:
        print(json.loads(line))


## Command-line alternative

The notebook and command line call the same driver. From the repository root, the equivalent command is:

```bash
/opt/anaconda3/envs/sports_net/bin/python \
  "3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis/code/EDUCATION_20260929_alba_origin_report_checks.py" \
  --run \
  --archive-use-acknowledged \
  --initial-interval 2.0 \
  --minimum-interval 1.2 \
  --maximum-interval 60 \
  --max-hours 6 \
  --retry-passes 3 \
  --retry-cooldown 180
```

The command is also resumable. Successful school codes are skipped on subsequent invocations.